# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [7]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [11]:
# 检查 Notebook 实际使用的 Python 和 PyTorch
import sys
import torch
print("Python 路径：", sys.executable)
print("PyTorch 版本：", torch.__version__)


Python 路径： C:\ProgramData\miniconda3\envs\torch-class\python.exe
PyTorch 版本： 2.7.1+cpu


In [12]:
import torch
# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

a = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
c = torch.tensor(0.0, requires_grad=True)
y_pred = a * x ** 2 + b * x + c
loss = torch.mean(torch.abs(y - y_pred))
print("初始参数：")
print(f"a = {a.item():.4f}, b = {b.item():.4f}, c = {c.item():.4f}")
print("初始平均绝对损失：", loss.item())


def train_quad(init_a=0.0, init_b=0.0, init_c=0.0, epochs=40000, base_lr=0.015):
    a = torch.tensor(float(init_a), requires_grad=True)
    b = torch.tensor(float(init_b), requires_grad=True)
    c = torch.tensor(float(init_c), requires_grad=True)
    optimizer = torch.optim.SGD([a, b, c], lr=base_lr, momentum=0.9)
    history = []
    
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        y_pred = a * x ** 2 + b * x + c
        loss = torch.mean(torch.abs(y - y_pred))
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            current_loss = torch.mean(torch.abs(y - (a * x**2 + b * x + c))).item()
        history.append((epoch, base_lr, a.item(), b.item(), c.item(), current_loss))
    return a.detach().item(), b.detach().item(), c.detach().item(), history


learned_a, learned_b, learned_c, history = train_quad()

print("\n轮数    学习率          a           b           c           MAE损失")
for idx in [0, 1, 4, 9, 99, 999, 10000, -1]:
    ep, lr, va, vb, vc, cl = history[idx]
    print(f"{ep:5d}  {lr:.6f}  {va:11.6f}  {vb:11.6f}  {vc:11.6f}  {cl:.6f}")

print(f"\n训练得到参数：\na={learned_a:.4f}, b={learned_b:.4f}, c={learned_c:.4f}")

with torch.no_grad():
    y_pred_final = learned_a * x ** 2 + learned_b * x + learned_c
    final_mae = torch.mean(torch.abs(y - y_pred_final))

print("\n==== 每个样本结果 ====")
for xi, yi, ypi in zip(x, y, y_pred_final):
    print(f"x={xi:4.1f}, 真实y={yi:6.3f}, 预测y={ypi:6.3f}")

print(f"\n最终平均绝对误差 MAE = {final_mae.item():.4f}")


初始参数：
a = 0.0000, b = 0.0000, c = 0.0000
初始平均绝对损失： 2.1388888359069824

轮数    学习率          a           b           c           MAE损失
    1  0.015000     0.002500     0.013333     0.006667  2.124120
    2  0.015000     0.008917     0.037000     0.021000  2.095968
    5  0.015000     0.047942     0.160173     0.102709  1.943932
   10  0.015000     0.161566     0.493634     0.333987  1.524950
  100  0.015000    -0.534075     1.476034     1.986509  0.070282
 1000  0.015000    -0.511080     1.470506     2.000501  0.032826
10001  0.015000    -0.535806     1.473456     1.983784  0.075892
40000  0.015000    -0.511080     1.470506     2.000501  0.032826

训练得到参数：
a=-0.5111, b=1.4705, c=2.0005

==== 每个样本结果 ====
x=-2.0, 真实y=-3.000, 预测y=-2.985
x=-1.5, 真实y=-1.375, 预测y=-1.355
x=-1.0, 真实y= 0.000, 预测y= 0.019
x=-0.5, 真实y= 1.125, 预测y= 1.137
x= 0.0, 真实y= 2.000, 预测y= 2.001
x= 0.5, 真实y= 2.625, 预测y= 2.608
x= 1.0, 真实y= 3.000, 预测y= 2.960
x= 1.5, 真实y= 3.125, 预测y= 3.056
x= 2.0, 真实y= 3.000, 预测y= 2.897

最终平均绝对误差 MA

### 我的观察与说明

请在这里填写你的回答:预测函数改变参数由一变三，优化难度加大
优化器需要传入a,b,c三个参数，训练记录里同时跟踪 a、b、c 三个参数的变化
如何判断这次训练有效？
用训练完成后的参数计算的预测值，和题目给出的真实标签 y 比较接近
训练出来的参数收敛到稳定值，不会无限制大幅度跳动
整体来看，随着迭代轮次增加，平均绝对损失大体呈下降趋势，最后稳定在一个比较小的数值，没有持续上涨或者完全不变

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。